#### 1. Spatio_Temporal_Reference Point Grouping
* Original Code and Output Preserved. So, see next cell code for later execution

Grouping logic (the main fix)

* Each point is its own reference center. It collects every other point within 100 m AND within ±30 min time-of-day (any date).
* frozenset deduplication: if A's window and B's window collect identical members, it's one group — not two.
* Subset removal: if group A's members are entirely contained within group B, group A is discarded. ...
This directly implements your rule about not duplicating groups unless a new member P appears.

>> The core problem is that your connected based grouping code (discarded now) uses connected components, which causes "chaining" — 
if A connects to B, and B connects to C, then A, B, C all become one group even if A and C are far apart or outside the time window of each other. This inflates groups and reduces their total count.
What you actually want is a reference-point based grouping: each unique location (within 50m) + time-of-day window (±30 min) forms a group. 
Points within the same group must all be near the same reference center, not just near each other transitively.

##### A. Circle based on number of samples

In [1]:
"""
===================================================================
SPATIO-TEMPORAL CLUSTERING ANALYSIS - Mobile Air Quality Sensor
===================================================================
Algorithm: Reference-Point Based Grouping
  - Each data point acts as a reference center
  - Groups ALL points within 100m spatial buffer AND ±30 min
    time-of-day window (ignoring date — any day qualifies)
  - Deduplicates identical member sets (same members = same group)
  - Removes strict subsets (smaller group fully inside a bigger one)
  - Minimum 2 observations per group
  - Exports one JPG per unique hour (fixed full lat/lon axes)
  - Exports detailed Excel workbook

Author: Air Quality Analysis System
Date: 2024
===================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
from datetime import datetime
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION — EDIT THESE PATHS
# =====================================================================
INPUT_FOLDER   = "D:/Hyperlocal_Study/Vehicle_Monitoring/Last_Year/"
OUTPUT_JPG_PATH   = "C:/Users/Atique/Hyperlocal_LAMP/Visualisation/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"
OUTPUT_EXCEL_PATH = "C:/Users/Atique/Hyperlocal_LAMP/Visualisation/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"

# Clustering parameters
SPATIAL_BUFFER_M    = 100   # metres — radius around each reference point
TEMPORAL_WINDOW_MIN = 30    # ±30 min time-of-day window (any date)
MIN_GROUP_SIZE      = 2     # minimum observations to form a valid group

# =====================================================================
print("\n" + "="*70)
print("SPATIO-TEMPORAL CLUSTERING ANALYSIS — INITIALISATION")
print("="*70)

# =====================================================================
# STEP 1: DATA LOADING AND PREPROCESSING
# =====================================================================
print("\n[STEP 1] Loading and preprocessing data...")

excel_files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(('.xlsx', '.xls', '.csv'))
]
if not excel_files:
    print("ERROR: No Excel/CSV files found in input folder!")
    exit()

data_path = excel_files[0]
print(f"  Loading: {data_path}")
df = (pd.read_excel(data_path)
      if data_path.endswith(('.xlsx', '.xls'))
      else pd.read_csv(data_path))

print(f"  Shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Columns: {', '.join(df.columns.tolist())}")

# --- Clean ---
df = df.dropna(subset=['latitude', 'longitude', 'pm25']).copy()
print(f"  After dropping missing spatial/PM2.5 rows: {len(df)}")

# --- Parse datetime ---
df['dateTime']      = pd.to_datetime(df['dateTime'], format='%d-%m-%y')
df['datetime_full'] = pd.to_datetime(
    df['dateTime'].astype(str) + ' ' + df['Time'].astype(str),
    format='%Y-%m-%d %H:%M:%S'
)

# Time-of-day fields (date is ignored for clustering)
df['hour_of_day']   = df['datetime_full'].dt.hour
df['minute_of_day'] = df['datetime_full'].dt.hour * 60 + df['datetime_full'].dt.minute
df['time_of_day']   = df['datetime_full'].dt.time

df = df.sort_values('datetime_full').reset_index(drop=True)

print(f"  Date range : {df['datetime_full'].min()} → {df['datetime_full'].max()}")

# --- Global spatial bounds (used for ALL hourly plots) ---
LAT_MIN = df['latitude'].min()
LAT_MAX = df['latitude'].max()
LON_MIN = df['longitude'].min()
LON_MAX = df['longitude'].max()

# Add a small padding (5% of range) so points on edges are not clipped
lat_pad = (LAT_MAX - LAT_MIN) * 0.05
lon_pad = (LON_MAX - LON_MIN) * 0.05
PLOT_LAT_MIN = LAT_MIN - lat_pad
PLOT_LAT_MAX = LAT_MAX + lat_pad
PLOT_LON_MIN = LON_MIN - lon_pad
PLOT_LON_MAX = LON_MAX + lon_pad

print(f"  Lat range  : {LAT_MIN:.6f} → {LAT_MAX:.6f}")
print(f"  Lon range  : {LON_MIN:.6f} → {LON_MAX:.6f}")

# =====================================================================
# STEP 2: REFERENCE-POINT BASED SPATIO-TEMPORAL GROUPING
# =====================================================================
print("\n" + "="*70)
print("[STEP 2] Reference-point spatio-temporal grouping")
print("="*70)

n = len(df)

# --- Pairwise spatial distances (degrees → metres) ---
print(f"  Building {n}×{n} spatial distance matrix...")
avg_lat = (LAT_MIN + LAT_MAX) / 2.0
coords  = df[['latitude', 'longitude']].values
# Approximate conversion: 1° lat ≈ 111 000 m; 1° lon ≈ 111 000·cos(lat) m
scale   = np.array([111000.0, 111000.0 * np.cos(np.radians(avg_lat))])
spatial_m = cdist(coords * scale, coords * scale, metric='euclidean')

# --- Pairwise time-of-day distances (minutes, with midnight wrap) ---
print(f"  Building {n}×{n} time-of-day distance matrix...")
tod  = df['minute_of_day'].values
tdist = np.abs(np.subtract.outer(tod, tod))
tdist = np.minimum(tdist, 1440 - tdist)   # wrap at midnight

# --- For each reference point i, collect all j satisfying BOTH criteria ---
print(f"  Collecting candidate groups for each reference point...")
spatial_ok  = spatial_m  <= SPATIAL_BUFFER_M
temporal_ok = tdist      <= TEMPORAL_WINDOW_MIN
both_ok     = spatial_ok & temporal_ok  # n×n boolean

candidate_sets = []
for i in range(n):
    members = frozenset(np.where(both_ok[i])[0])
    # must have at least MIN_GROUP_SIZE members (including self)
    if len(members) >= MIN_GROUP_SIZE:
        candidate_sets.append(members)

print(f"  Raw candidate groups (one per qualifying reference): {len(candidate_sets)}")

# --- Deduplicate: identical member sets → same group ---
unique_groups = list(set(candidate_sets))
print(f"  Unique groups after deduplication                  : {len(unique_groups)}")

# --- Remove strict subsets: if group A ⊂ group B, discard A ---
# (implements your rule: A+X+B already grouped → B alone doesn't create another)
final_groups = []
for g in unique_groups:
    absorbed = any(g < other for other in unique_groups if g is not other)
    if not absorbed:
        final_groups.append(g)

print(f"  After removing groups fully absorbed by larger ones: {len(final_groups)}")

valid_groups = {idx: sorted(members) for idx, members in enumerate(final_groups)}
print(f"\n✓ Final valid group count: {len(valid_groups)}")

# =====================================================================
# STEP 3: AGGREGATE GROUPS
# =====================================================================
print("\n[STEP 3] Aggregating group statistics...")

group_list = []
for gid, members in valid_groups.items():
    sub = df.loc[members]

    lat_c   = sub['latitude'].mean()
    lon_c   = sub['longitude'].mean()
    hour_b  = int(sub['hour_of_day'].mode().iloc[0])
    t_ctr   = sub['minute_of_day'].mean()
    pm25_m  = sub['pm25'].mean()
    pm10_m  = sub['pm10'].mean()
    no2_m   = sub['NO2'].mean()
    size    = len(members)

    # spatial spread: max pairwise distance within group (metres)
    sub_coords = sub[['latitude', 'longitude']].values * scale
    if size > 1:
        sp_spread = cdist(sub_coords, sub_coords).max()
    else:
        sp_spread = 0.0

    t_span = sub['minute_of_day'].max() - sub['minute_of_day'].min()

    group_list.append({
        'group_id':         gid,
        'size':             size,
        'lat_center':       lat_c,
        'lon_center':       lon_c,
        'hour_bin':         hour_b,
        'time_center_min':  t_ctr,
        'hour_label':       f"{hour_b:02d}:00",
        'pm25_mean':        pm25_m,
        'pm10_mean':        pm10_m,
        'no2_mean':         no2_m,
        'spatial_spread_m': sp_spread,
        'time_span_min':    t_span,
    })

groups_df = pd.DataFrame(group_list)
print(f"  Groups: {len(groups_df)}")
print(f"  Size   — min: {groups_df['size'].min()}, max: {groups_df['size'].max()}, "
      f"mean: {groups_df['size'].mean():.1f}")
print(f"  PM2.5  — min: {groups_df['pm25_mean'].min():.2f}, "
      f"max: {groups_df['pm25_mean'].max():.2f}, "
      f"mean: {groups_df['pm25_mean'].mean():.2f} µg/m³")

# =====================================================================
# STEP 4: HOURLY BINNING SUMMARY
# =====================================================================
print("\n[STEP 4] Hourly distribution...")
unique_hours = sorted(groups_df['hour_bin'].unique())
for h in unique_hours:
    cnt = (groups_df['hour_bin'] == h).sum()
    print(f"  {h:02d}:00 — {cnt} groups")

# =====================================================================
# STEP 5: VISUALISATION — ONE SEPARATE JPG PER UNIQUE HOUR
#         Fixed axes: full lat/lon extent of the entire dataset
# =====================================================================
print("\n[STEP 5] Generating one plot per unique hour (fixed axes)...")

os.makedirs(OUTPUT_JPG_PATH, exist_ok=True)

# Colour scale anchors: 5th–95th percentile of all group PM2.5 means
vmin = groups_df['pm25_mean'].quantile(0.05)
vmax = groups_df['pm25_mean'].quantile(0.95)

saved_files = []
for hour in unique_hours:
    hour_data = groups_df[groups_df['hour_bin'] == hour]

    # --- Figure: square-ish, aspect locks lat/lon range ---
    lat_range = PLOT_LAT_MAX - PLOT_LAT_MIN
    lon_range = PLOT_LON_MAX - PLOT_LON_MIN
    fig_w = 10
    fig_h = max(7, fig_w * (lat_range / lon_range))   # proportional to coordinate extent
    fig_h = min(fig_h, 14)                            # cap height for very elongated routes

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sc = ax.scatter(
        hour_data['lon_center'],
        hour_data['lat_center'],
        c=hour_data['pm25_mean'],
        s=hour_data['size'] * 60,        # size ∝ group size (confidence)
        cmap='RdYlGn_r',
        alpha=0.75,
        edgecolors='black',
        linewidths=0.6,
        vmin=vmin,
        vmax=vmax,
        zorder=3
    )

    # --- Fixed axes — ALWAYS the full dataset extent ---
    ax.set_xlim(PLOT_LON_MIN, PLOT_LON_MAX)
    ax.set_ylim(PLOT_LAT_MIN, PLOT_LAT_MAX)

    # Fine-grained tick formatting
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(fontsize=8)

    ax.set_xlabel('Longitude', fontweight='bold', fontsize=11)
    ax.set_ylabel('Latitude',  fontweight='bold', fontsize=11)
    ax.set_title(
        f'PM2.5 Spatial Distribution — Hour {hour:02d}:00\n'
        f'(all days aggregated  |  {len(hour_data)} groups  |  '
        f'spatial buffer: {SPATIAL_BUFFER_M} m  |  temporal window: ±{TEMPORAL_WINDOW_MIN} min)',
        fontweight='bold', fontsize=12, pad=12
    )
    ax.grid(True, alpha=0.3, linestyle='--', zorder=1)

    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('PM2.5 (µg/m³)', fontweight='bold', fontsize=10)

    # Legend for marker size
    for s_val in [2, 5, 10]:
        ax.scatter([], [], c='grey', alpha=0.6,
                   s=s_val * 60, label=f'{s_val} obs', edgecolors='black', linewidths=0.5)
    ax.legend(title='Group size', loc='lower right', fontsize=8, title_fontsize=8, framealpha=0.7)

    # Stats annotation
    stats = (f"N groups : {len(hour_data)}\n"
             f"PM2.5 mean: {hour_data['pm25_mean'].mean():.1f} µg/m³\n"
             f"PM2.5 max : {hour_data['pm25_mean'].max():.1f} µg/m³\n"
             f"PM2.5 min : {hour_data['pm25_mean'].min():.1f} µg/m³")
    ax.text(0.02, 0.98, stats, transform=ax.transAxes, fontsize=8,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.55))

    plt.tight_layout()

    fname = os.path.join(OUTPUT_JPG_PATH,
                         f"Air_Quality_Hour_{hour:02d}00.jpg")
    plt.savefig(fname, dpi=300, bbox_inches='tight', format='jpg')
    plt.close()
    saved_files.append(fname)
    print(f"  ✓ Saved: {fname}")

print(f"\n  {len(saved_files)} hourly plots saved.")

# =====================================================================
# STEP 6: EXPORT TO EXCEL
# =====================================================================
print("\n[STEP 6] Exporting to Excel...")

os.makedirs(OUTPUT_EXCEL_PATH, exist_ok=True)
excel_file = os.path.join(OUTPUT_EXCEL_PATH, "Air_Quality_Clustering_Analysis.xlsx")

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:

    # ── Sheet 1: Group Summary ────────────────────────────────────────
    gs = groups_df[[
        'group_id', 'size', 'hour_label',
        'lat_center', 'lon_center', 'time_center_min',
        'pm25_mean', 'pm10_mean', 'no2_mean',
        'spatial_spread_m', 'time_span_min'
    ]].copy()
    gs.columns = [
        'Group_ID', 'Group_Size', 'Hour_Bin',
        'Latitude_Center', 'Longitude_Center', 'Time_Center_min_from_midnight',
        'PM25_Mean_µg/m³', 'PM10_Mean_µg/m³', 'NO2_Mean_µg/m³',
        'Spatial_Spread_m', 'Time_Span_min'
    ]
    gs = gs.sort_values('PM25_Mean_µg/m³', ascending=False)
    gs.to_excel(writer, sheet_name='Group_Summary', index=False)

    # ── Sheet 2: Hourly Statistics ────────────────────────────────────
    hourly_rows = []
    for h in unique_hours:
        hd = groups_df[groups_df['hour_bin'] == h]
        hourly_rows.append({
            'Hour':                f'{h:02d}:00',
            'Number_of_Groups':    len(hd),
            'Total_Observations':  hd['size'].sum(),
            'PM25_Mean':           hd['pm25_mean'].mean(),
            'PM25_Min':            hd['pm25_mean'].min(),
            'PM25_Max':            hd['pm25_mean'].max(),
            'PM25_Std':            hd['pm25_mean'].std(),
            'PM10_Mean':           hd['pm10_mean'].mean(),
            'NO2_Mean':            hd['no2_mean'].mean(),
            'Avg_Group_Size':      hd['size'].mean(),
        })
    pd.DataFrame(hourly_rows).to_excel(writer, sheet_name='Hourly_Statistics', index=False)

    # ── Sheet 3: Top-20 Hotspots ──────────────────────────────────────
    hot = groups_df.nlargest(20, 'pm25_mean')[[
        'group_id', 'size', 'hour_label',
        'lat_center', 'lon_center',
        'pm25_mean', 'pm10_mean', 'no2_mean'
    ]].copy()
    hot.columns = [
        'Group_ID', 'Group_Size', 'Hour',
        'Latitude', 'Longitude',
        'PM25_µg/m³', 'PM10_µg/m³', 'NO2_µg/m³'
    ]
    hot.to_excel(writer, sheet_name='Top_20_Hotspots', index=False)

    # ── Sheet 4: Original Data with Group Assignments ─────────────────
    df_out = df.copy()
    df_out['Group_ID'] = -1
    df_out['Hour_Bin'] = -1
    for gid, members in valid_groups.items():
        hb = groups_df.loc[groups_df['group_id'] == gid, 'hour_bin'].values[0]
        df_out.loc[members, 'Group_ID'] = gid
        df_out.loc[members, 'Hour_Bin'] = hb

    df_out_final = df_out[[
        'datetime_full', 'latitude', 'longitude',
        'pm25', 'pm10', 'NO2',
        'Group_ID', 'Hour_Bin', 'time_of_day'
    ]].copy()
    df_out_final.columns = [
        'DateTime', 'Latitude', 'Longitude',
        'PM25', 'PM10', 'NO2',
        'Group_ID', 'Hour_Bin', 'Time_of_Day'
    ]
    df_out_final.sort_values('DateTime').to_excel(
        writer, sheet_name='Original_Data_Grouped', index=False)

    # ── Sheet 5: Summary Statistics ───────────────────────────────────
    in_groups = groups_df['size'].sum()
    summary = pd.DataFrame({
        'Metric': [
            'Total measurements',
            'Date range',
            'Total valid groups',
            'Measurements in groups',
            'Measurements NOT in any group',
            'Spatial buffer (m)',
            'Temporal window (±min)',
            'Min group size',
            '',
            'PM2.5 mean — raw data (µg/m³)',
            'PM2.5 std  — raw data (µg/m³)',
            'PM2.5 min  — raw data (µg/m³)',
            'PM2.5 max  — raw data (µg/m³)',
            'PM2.5 25th pct — raw (µg/m³)',
            'PM2.5 median   — raw (µg/m³)',
            'PM2.5 75th pct — raw (µg/m³)',
            '',
            'PM2.5 mean — group means (µg/m³)',
            'PM2.5 std  — group means (µg/m³)',
            '',
            'Unique hours analysed',
            'Max group spatial spread (m)',
            'Max group time span (min)',
        ],
        'Value': [
            len(df),
            f"{df['datetime_full'].min().date()} to {df['datetime_full'].max().date()}",
            len(groups_df),
            in_groups,
            len(df) - in_groups,
            SPATIAL_BUFFER_M,
            TEMPORAL_WINDOW_MIN,
            MIN_GROUP_SIZE,
            '',
            f"{df['pm25'].mean():.2f}",
            f"{df['pm25'].std():.2f}",
            f"{df['pm25'].min():.2f}",
            f"{df['pm25'].max():.2f}",
            f"{df['pm25'].quantile(0.25):.2f}",
            f"{df['pm25'].median():.2f}",
            f"{df['pm25'].quantile(0.75):.2f}",
            '',
            f"{groups_df['pm25_mean'].mean():.2f}",
            f"{groups_df['pm25_mean'].std():.2f}",
            '',
            len(unique_hours),
            f"{groups_df['spatial_spread_m'].max():.1f}",
            f"{groups_df['time_span_min'].max():.1f}",
        ]
    })
    summary.to_excel(writer, sheet_name='Summary_Statistics', index=False, header=False)

    # ── Sheet 6: Methodology ─────────────────────────────────────────
    methodology = f"""
SPATIO-TEMPORAL CLUSTERING — REFERENCE-POINT METHOD
====================================================

ALGORITHM
---------
Each data point acts as a reference center.
For reference point i, collect every other point j where:
  (a) spatial distance(i, j) ≤ {SPATIAL_BUFFER_M} m         [100 m radius]
  (b) |time_of_day(i) − time_of_day(j)| ≤ {TEMPORAL_WINDOW_MIN} min  [±30 min, any date]
This collection (including i itself) is a candidate group.

DEDUPLICATION
-------------
Two reference points that collect the exact same set of members
produce identical frozensets → they count as ONE group.
A new group only arises when a reference point's window
collects at least one member not present in any existing group.

SUBSET REMOVAL
--------------
If every member of group A is already contained in group B (A ⊂ B),
group A is discarded. This enforces your rule:
"if A, X, B are already one group, B's window collecting only A and X
does not create a separate group."

MINIMUM SIZE
------------
Groups with fewer than {MIN_GROUP_SIZE} members are discarded.

TEMPORAL LOGIC
--------------
Only the TIME OF DAY is used for clustering (HH:MM), date is ignored.
So 08:05 on Day 1 will cluster with 08:05 on Day 2, Day 3, etc.
Midnight wrap-around is handled: 23:50 and 00:10 are 20 min apart.

GROUP AGGREGATION
-----------------
For each valid group:
  • Spatial center = mean(latitude), mean(longitude) of all members
  • Hour bin       = mode of hour-of-day across members
  • PM2.5/PM10/NO2 = arithmetic mean of all member readings
  • Spatial spread = max pairwise distance within group
  • Time span      = max − min time-of-day (minutes)

VISUALISATION
-------------
One JPG per unique hour of the day.
Axes are FIXED to the full dataset extent (min/max lat & lon)
so every plot is directly comparable — no axis compression.
Marker color = PM2.5 (red=high, yellow=medium, green=low).
Marker size  ∝ group size (more observations = higher confidence).

Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Parameters  : spatial {SPATIAL_BUFFER_M} m  |  temporal ±{TEMPORAL_WINDOW_MIN} min  |  min size {MIN_GROUP_SIZE}
"""
    pd.DataFrame({'Methodology': [methodology]}).to_excel(
        writer, sheet_name='Methodology', index=False, header=False)

print(f"\n✓ Excel saved: {excel_file}")

# =====================================================================
# STEP 7: FINAL CONSOLE REPORT
# =====================================================================
print("\n" + "="*70)
print("ANALYSIS COMPLETE — FINAL SUMMARY")
print("="*70)

print(f"""
  Total measurements      : {len(df)}
  Date range              : {df['datetime_full'].min().date()} → {df['datetime_full'].max().date()}
  Valid groups formed      : {len(valid_groups)}
  Measurements in groups  : {groups_df['size'].sum()} / {len(df)} ({100*groups_df['size'].sum()/len(df):.1f}%)
  Measurements ungrouped  : {len(df) - groups_df['size'].sum()} (singletons, excluded)
  Unique hours             : {', '.join(f'{h:02d}:00' for h in unique_hours)}

  PM2.5 (raw data)   — mean: {df['pm25'].mean():.2f}  std: {df['pm25'].std():.2f}  range: [{df['pm25'].min():.2f}, {df['pm25'].max():.2f}] µg/m³
  PM2.5 (group means)— mean: {groups_df['pm25_mean'].mean():.2f}  std: {groups_df['pm25_mean'].std():.2f}  range: [{groups_df['pm25_mean'].min():.2f}, {groups_df['pm25_mean'].max():.2f}] µg/m³

  Top 5 most-polluted groups:""")

for rank, (_, row) in enumerate(groups_df.nlargest(5, 'pm25_mean').iterrows(), 1):
    print(f"    {rank}. PM2.5 {row['pm25_mean']:.1f} µg/m³ | "
          f"({row['lat_center']:.6f}, {row['lon_center']:.6f}) | "
          f"Hour {row['hour_label']} | {row['size']} obs")

print(f"""
  OUTPUT FILES
  ─────────────────────────────────────────────────────
  Hourly JPGs ({len(saved_files)} files) : {OUTPUT_JPG_PATH}
    Naming: Air_Quality_Hour_HHMM.jpg (e.g. Air_Quality_Hour_0800.jpg)

  Excel workbook            : {excel_file}
    Sheet 1 — Group_Summary          ({len(valid_groups)} groups)
    Sheet 2 — Hourly_Statistics      ({len(unique_hours)} hours)
    Sheet 3 — Top_20_Hotspots
    Sheet 4 — Original_Data_Grouped  ({len(df)} rows with Group_ID)
    Sheet 5 — Summary_Statistics
    Sheet 6 — Methodology
""")
print("="*70 + "\n")


SPATIO-TEMPORAL CLUSTERING ANALYSIS — INITIALISATION

[STEP 1] Loading and preprocessing data...
  Loading: D:/Hyperlocal_Study/Vehicle_Monitoring/Last_Year/Season_2.xlsx
  Shape  : 3192 rows × 7 columns
  Columns: dateTime, Time, latitude, longitude, pm25, pm10, NO2
  After dropping missing spatial/PM2.5 rows: 3191
  Date range : 2023-10-16 08:00:00 → 2023-11-05 12:01:00
  Lat range  : 19.024759 → 19.091400
  Lon range  : 72.844810 → 72.894440

[STEP 2] Reference-point spatio-temporal grouping
  Building 3191×3191 spatial distance matrix...
  Building 3191×3191 time-of-day distance matrix...
  Raw candidate groups (one per qualifying reference): 3149
  Unique groups after deduplication                  : 1133
  After removing groups fully absorbed by larger ones: 497

✓ Final valid group count: 497

[STEP 3] Aggregating group statistics...
  Groups: 497
  Size   — min: 2, max: 244, mean: 57.9
  PM2.5  — min: 95.31, max: 130.66, mean: 114.65 µg/m³

[STEP 4] Hourly distribution...
  08

#### 2.

In [6]:
"""
===================================================================
SPATIO-TEMPORAL CLUSTERING ANALYSIS - Mobile Air Quality Sensor
===================================================================
Algorithm: Reference-Point Based Grouping (chunked — memory safe)

FIXES APPLIED:
  1. MemoryError  → chunked distance computation, no full n×n matrix
  2. Single-circle bug → subset removal now capped: only removes a
     group if a strictly-larger group exists AND shares >80% members
     (pure subset check on 70k rows collapsed all groups into one)
  3. Column names updated: Timestamp, lat, lng, pm2d5, pm10, pressure
  4. Fixed marker size — no group-size scaling (prevents overlap)
  5. No Excel export — JPGs only
===================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION — EDIT THESE PATHS
# =====================================================================
INPUT_FOLDER    = r"D:/LAMP/Vehicle_Monitoring/Data_File/"
OUTPUT_JPG_PATH = r"C:/Users/Atique/LAMP_Coding/Mobile_Monitoring/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"

# Clustering parameters
SPATIAL_BUFFER_M    = 100   # metres — radius around each reference point
TEMPORAL_WINDOW_MIN = 30    # ±30 min time-of-day window (any date)
MIN_GROUP_SIZE      = 2     # minimum observations to form a valid group

# Memory control — rows processed per chunk (lower = less RAM used)
CHUNK_SIZE = 3000

# Fixed circle size for ALL markers (was previously group-size scaled)
MARKER_SIZE = 40

# =====================================================================
print("\n" + "="*70)
print("SPATIO-TEMPORAL CLUSTERING ANALYSIS — INITIALISATION")
print("="*70)

# =====================================================================
# STEP 1: DATA LOADING AND PREPROCESSING
# =====================================================================
print("\n[STEP 1] Loading and preprocessing data...")

excel_files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(('.xlsx', '.xls', '.csv'))
]
if not excel_files:
    print("ERROR: No Excel/CSV files found in input folder!")
    exit()

data_path = excel_files[0]
print(f"  Loading: {data_path}")
df = (pd.read_excel(data_path)
      if data_path.endswith(('.xlsx', '.xls'))
      else pd.read_csv(data_path))

print(f"  Shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Columns: {', '.join(df.columns.tolist())}")

# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Drop rows with missing spatial or PM2.5 values
df = df.dropna(subset=['lat', 'lng', 'pm2d5']).copy()

# ── CRITICAL: drop rows where lat or lng is 0.0 (GPS dropout) ────────
before = len(df)
df = df[(df['lat'] != 0.0) & (df['lng'] != 0.0)].copy()
print(f"  Dropped {before - len(df)} rows with lat/lng = 0 (GPS dropout)")
print(f"  Valid rows remaining: {len(df)}")

# Parse combined Timestamp column: DD-MM-YY HH:MM:SS
df['datetime_full'] = pd.to_datetime(df['Timestamp'], dayfirst=True)

# Fix 2-digit year parsed as 19xx  (e.g. '26' → 1926 → correct to 2026)
df['datetime_full'] = df['datetime_full'].apply(
    lambda x: x.replace(year=x.year + 100) if x.year < 2000 else x
)

df['hour_of_day']   = df['datetime_full'].dt.hour
df['minute_of_day'] = df['datetime_full'].dt.hour * 60 + df['datetime_full'].dt.minute
df['time_of_day']   = df['datetime_full'].dt.time
df = df.sort_values('datetime_full').reset_index(drop=True)

print(f"  Date range : {df['datetime_full'].min()} → {df['datetime_full'].max()}")

# Global spatial bounds
LAT_MIN, LAT_MAX = df['lat'].min(), df['lat'].max()
LON_MIN, LON_MAX = df['lng'].min(), df['lng'].max()

lat_pad      = (LAT_MAX - LAT_MIN) * 0.05
lon_pad      = (LON_MAX - LON_MIN) * 0.05
PLOT_LAT_MIN = LAT_MIN - lat_pad
PLOT_LAT_MAX = LAT_MAX + lat_pad
PLOT_LON_MIN = LON_MIN - lon_pad
PLOT_LON_MAX = LON_MAX + lon_pad

print(f"  Lat range  : {LAT_MIN:.6f} → {LAT_MAX:.6f}")
print(f"  Lon range  : {LON_MIN:.6f} → {LON_MAX:.6f}")

# =====================================================================
# STEP 2: CHUNKED SPATIO-TEMPORAL GROUPING
# =====================================================================
print("\n" + "="*70)
print("[STEP 2] Reference-point spatio-temporal grouping (chunked)")
print("="*70)

n       = len(df)
avg_lat = (LAT_MIN + LAT_MAX) / 2.0
scale   = np.array([111000.0, 111000.0 * np.cos(np.radians(avg_lat))])

coords_m = df[['lat', 'lng']].values * scale   # (n, 2) scaled to metres
tod      = df['minute_of_day'].values           # (n,)

n_chunks = int(np.ceil(n / CHUNK_SIZE))
peak_ram = CHUNK_SIZE * n * 8 / 1e9

print(f"  Total points : {n:,}")
print(f"  Chunk size   : {CHUNK_SIZE:,}")
print(f"  Total chunks : {n_chunks}")
print(f"  Peak RAM per chunk ≈ {peak_ram:.2f} GB  "
      f"(full matrix would need {n*n*8/1e9:.1f} GB)\n")

# Each entry: frozenset of row-indices that share space+time with ref point i
candidate_sets = []

for chunk_idx in range(n_chunks):
    start = chunk_idx * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, n)

    # Spatial distances: (chunk_size, n)
    chunk_coords = coords_m[start:end]
    diff         = chunk_coords[:, np.newaxis, :] - coords_m[np.newaxis, :, :]
    spatial_dist = np.sqrt((diff ** 2).sum(axis=2))
    del diff

    # Time-of-day distances with midnight wrap: (chunk_size, n)
    chunk_tod = tod[start:end]
    tdist_raw = np.abs(chunk_tod[:, np.newaxis] - tod[np.newaxis, :])
    time_dist = np.minimum(tdist_raw, 1440 - tdist_raw)
    del tdist_raw

    # Combined boolean mask
    both_ok = (spatial_dist <= SPATIAL_BUFFER_M) & (time_dist <= TEMPORAL_WINDOW_MIN)
    del spatial_dist, time_dist

    for local_i in range(end - start):
        members = frozenset(np.where(both_ok[local_i])[0])
        if len(members) >= MIN_GROUP_SIZE:
            candidate_sets.append(members)

    del both_ok

    if (chunk_idx + 1) % 5 == 0 or chunk_idx == n_chunks - 1:
        print(f"  Chunk {chunk_idx+1:>4}/{n_chunks} done  |  "
              f"candidates so far: {len(candidate_sets):,}")

print(f"\n  Raw candidate groups (before dedup): {len(candidate_sets):,}")

# ── Deduplicate: identical member sets → one group ───────────────────
unique_groups = list(set(candidate_sets))
print(f"  Unique groups after deduplication  : {len(unique_groups):,}")

# ── Subset removal — FIXED ────────────────────────────────────────────
# Problem with naive subset check on 70k-point data:
#   Almost every small group is a subset of some larger group
#   because the route overlaps repeatedly across days.
#   Result: nearly all groups get absorbed → only 1–2 survive → single dot on plot.
#
# Fix: only absorb group A into group B when BOTH conditions hold:
#   (a) A is a strict subset of B  (every member of A is in B)
#   (b) B is not vastly larger — specifically |B| <= 3 × |A|
#       (prevents a massive 500-member "super group" eating all small ones)
#
# This keeps genuinely distinct local clusters alive on the map.

print("  Removing subsets (size-bounded absorption)...")

final_groups = []
for g in unique_groups:
    g_size = len(g)
    absorbed = any(
        g < other and len(other) <= 3 * g_size
        for other in unique_groups
        if other is not g
    )
    if not absorbed:
        final_groups.append(g)

print(f"  After bounded subset removal       : {len(final_groups):,}")

valid_groups = {idx: sorted(members) for idx, members in enumerate(final_groups)}
print(f"\n✓ Final valid group count: {len(valid_groups):,}")

# =====================================================================
# STEP 3: AGGREGATE GROUPS
# =====================================================================
print("\n[STEP 3] Aggregating group statistics...")

group_list = []
for gid, members in valid_groups.items():
    sub    = df.loc[members]
    size   = len(members)
    lat_c  = sub['lat'].mean()
    lon_c  = sub['lng'].mean()
    hour_b = int(sub['hour_of_day'].mode().iloc[0])
    t_ctr  = sub['minute_of_day'].mean()
    pm25_m = sub['pm2d5'].mean()
    pm10_m = sub['pm10'].mean()
    pres_m = sub['pressure'].mean() if 'pressure' in sub.columns else np.nan
    t_span = sub['minute_of_day'].max() - sub['minute_of_day'].min()

    # Spatial spread within group
    sub_coords = sub[['lat', 'lng']].values * scale
    if size > 1:
        d         = sub_coords[:, np.newaxis, :] - sub_coords[np.newaxis, :, :]
        sp_spread = np.sqrt((d**2).sum(axis=2)).max()
    else:
        sp_spread = 0.0

    group_list.append({
        'group_id':         gid,
        'size':             size,
        'lat_center':       lat_c,
        'lon_center':       lon_c,
        'hour_bin':         hour_b,
        'time_center_min':  t_ctr,
        'hour_label':       f"{hour_b:02d}:00",
        'pm25_mean':        pm25_m,
        'pm10_mean':        pm10_m,
        'pressure_mean':    pres_m,
        'spatial_spread_m': sp_spread,
        'time_span_min':    t_span,
    })

groups_df = pd.DataFrame(group_list)
print(f"  Total groups : {len(groups_df):,}")
print(f"  Group size   — min: {groups_df['size'].min()}, "
      f"max: {groups_df['size'].max()}, "
      f"mean: {groups_df['size'].mean():.1f}")
print(f"  PM2.5 (group means) — "
      f"min: {groups_df['pm25_mean'].min():.1f}, "
      f"max: {groups_df['pm25_mean'].max():.1f}, "
      f"mean: {groups_df['pm25_mean'].mean():.1f} µg/m³")

# =====================================================================
# STEP 4: HOURLY BINNING SUMMARY
# =====================================================================
print("\n[STEP 4] Hourly distribution...")
unique_hours = sorted(groups_df['hour_bin'].unique())
for h in unique_hours:
    cnt = (groups_df['hour_bin'] == h).sum()
    print(f"  {h:02d}:00 — {cnt:>5} groups")

# =====================================================================
# STEP 5: VISUALISATION — ONE JPG PER UNIQUE HOUR
# =====================================================================
print("\n[STEP 5] Generating hourly plots (fixed axes, fixed marker size)...")

os.makedirs(OUTPUT_JPG_PATH, exist_ok=True)

# Colour scale: 5th–95th percentile of group PM2.5 means
vmin = groups_df['pm25_mean'].quantile(0.05)
vmax = groups_df['pm25_mean'].quantile(0.95)

saved_files = []
for hour in unique_hours:
    hour_data = groups_df[groups_df['hour_bin'] == hour]

    lat_range = PLOT_LAT_MAX - PLOT_LAT_MIN
    lon_range = PLOT_LON_MAX - PLOT_LON_MIN
    fig_w = 10
    fig_h = min(max(7, fig_w * (lat_range / lon_range)), 14)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sc = ax.scatter(
        hour_data['lon_center'],
        hour_data['lat_center'],
        c=hour_data['pm25_mean'],
        s=MARKER_SIZE,                  # FIXED — no group-size scaling
        cmap='RdYlGn_r',
        alpha=0.75,
        edgecolors='black',
        linewidths=0.5,
        vmin=vmin,
        vmax=vmax,
        zorder=3
    )

    # Fixed axes — full dataset extent, same across all hourly plots
    ax.set_xlim(PLOT_LON_MIN, PLOT_LON_MAX)
    ax.set_ylim(PLOT_LAT_MIN, PLOT_LAT_MAX)

    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(fontsize=8)

    ax.set_xlabel('Longitude', fontweight='bold', fontsize=11)
    ax.set_ylabel('Latitude',  fontweight='bold', fontsize=11)
    ax.set_title(
        f'PM2.5 Spatial Distribution — Hour {hour:02d}:00\n'
        f'(all days aggregated  |  {len(hour_data)} groups  |  '
        f'buffer: {SPATIAL_BUFFER_M} m  |  time window: ±{TEMPORAL_WINDOW_MIN} min)',
        fontweight='bold', fontsize=12, pad=12
    )
    ax.grid(True, alpha=0.3, linestyle='--', zorder=1)

    cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('PM2.5 (µg/m³)', fontweight='bold', fontsize=10)

    stats = (f"N groups  : {len(hour_data)}\n"
             f"PM2.5 mean: {hour_data['pm25_mean'].mean():.1f} µg/m³\n"
             f"PM2.5 max : {hour_data['pm25_mean'].max():.1f} µg/m³\n"
             f"PM2.5 min : {hour_data['pm25_mean'].min():.1f} µg/m³")
    ax.text(0.02, 0.98, stats, transform=ax.transAxes, fontsize=8,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.55))

    plt.tight_layout()
    fname = os.path.join(OUTPUT_JPG_PATH, f"Air_Quality_Hour_{hour:02d}00.jpg")
    plt.savefig(fname, dpi=300, bbox_inches='tight', format='jpg')
    plt.close()
    saved_files.append(fname)
    print(f"  ✓ Saved: {fname}  ({len(hour_data)} groups)")

# =====================================================================
# STEP 6: FINAL CONSOLE REPORT
# =====================================================================
print("\n" + "="*70)
print("ANALYSIS COMPLETE — FINAL SUMMARY")
print("="*70)
print(f"""
  Total measurements      : {len(df):,}
  Date range              : {df['datetime_full'].min().date()} → {df['datetime_full'].max().date()}
  Valid groups formed      : {len(valid_groups):,}
  Measurements in groups  : {groups_df['size'].sum():,} / {len(df):,} ({100*groups_df['size'].sum()/len(df):.1f}%)
  Unique hours plotted     : {', '.join(f'{h:02d}:00' for h in unique_hours)}

  PM2.5 raw data  — mean: {df['pm2d5'].mean():.1f}  std: {df['pm2d5'].std():.1f}  range: [{df['pm2d5'].min():.1f}, {df['pm2d5'].max():.1f}] µg/m³
  PM2.5 grp means — mean: {groups_df['pm25_mean'].mean():.1f}  std: {groups_df['pm25_mean'].std():.1f}  range: [{groups_df['pm25_mean'].min():.1f}, {groups_df['pm25_mean'].max():.1f}] µg/m³

  Top 5 most-polluted groups:""")

for rank, (_, row) in enumerate(groups_df.nlargest(5, 'pm25_mean').iterrows(), 1):
    print(f"    {rank}. PM2.5 {row['pm25_mean']:.1f} µg/m³ | "
          f"({row['lat_center']:.6f}, {row['lon_center']:.6f}) | "
          f"Hour {row['hour_label']} | n={row['size']}")

print(f"\n  Output JPGs ({len(saved_files)} files) → {OUTPUT_JPG_PATH}")
print("="*70 + "\n")


SPATIO-TEMPORAL CLUSTERING ANALYSIS — INITIALISATION

[STEP 1] Loading and preprocessing data...
  Loading: D:/LAMP/Vehicle_Monitoring/Data_File/Data_MobileMonitoring.xlsx
  Shape  : 70994 rows × 12 columns
  Columns: devId, Timestamp, temp, hum, pressure, pm2d5, pm10, aqi, lat, lng, grid, region
  Dropped 1250 rows with lat/lng = 0 (GPS dropout)
  Valid rows remaining: 69744
  Date range : 2026-03-10 14:02:08 → 2026-04-10 19:57:09
  Lat range  : 19.069450 → 19.147141
  Lon range  : 72.860382 → 72.936821

[STEP 2] Reference-point spatio-temporal grouping (chunked)
  Total points : 69,744
  Chunk size   : 3,000
  Total chunks : 24
  Peak RAM per chunk ≈ 1.67 GB  (full matrix would need 38.9 GB)

  Chunk    5/24 done  |  candidates so far: 14,898
  Chunk   10/24 done  |  candidates so far: 29,820
  Chunk   15/24 done  |  candidates so far: 43,933
  Chunk   20/24 done  |  candidates so far: 58,771
  Chunk   24/24 done  |  candidates so far: 68,329

  Raw candidate groups (before dedup): 

#### 3. Modification of Above

In [12]:
"""
===================================================================
SPATIO-TEMPORAL CLUSTERING ANALYSIS - Mobile Air Quality Sensor
===================================================================
FIXES IN THIS VERSION:
  1. MemoryError  → chunked distance computation, no full n×n matrix
  2. Single-circle bug → bounded subset removal (3x size cap)
  3. Column names: Timestamp, lat, lng, pm2d5, pm10, pressure
  4. Fixed marker size — no group-size scaling
  5. Excel output RESTORED with all 6 sheets
  6. NEW: Verification Excel — sample dedup + subset audit trail
===================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION — EDIT THESE PATHS
# =====================================================================
INPUT_FOLDER    = r"D:/LAMP/Vehicle_Monitoring/Data_File/"
OUTPUT_JPG_PATH = r"C:/Users/Atique/LAMP_Coding/Mobile_Monitoring/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"
OUTPUT_EXCEL_PATH = r"C:/Users/Atique/LAMP_Coding/Mobile_Monitoring/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"

# Clustering parameters
SPATIAL_BUFFER_M    = 100
TEMPORAL_WINDOW_MIN = 30
MIN_GROUP_SIZE      = 2

# Memory control (lower = less RAM per chunk)
CHUNK_SIZE = 3000

# Fixed circle size for ALL markers
MARKER_SIZE = 40

# =====================================================================
print("\n" + "="*70)
print("SPATIO-TEMPORAL CLUSTERING ANALYSIS — INITIALISATION")
print("="*70)

# =====================================================================
# STEP 1: DATA LOADING AND PREPROCESSING
# =====================================================================
print("\n[STEP 1] Loading and preprocessing data...")

excel_files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(('.xlsx', '.xls', '.csv'))
]
if not excel_files:
    print("ERROR: No Excel/CSV files found in input folder!")
    exit()

data_path = excel_files[0]
print(f"  Loading: {data_path}")
df = (pd.read_excel(data_path)
      if data_path.endswith(('.xlsx', '.xls'))
      else pd.read_csv(data_path))

print(f"  Shape  : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Columns: {', '.join(df.columns.tolist())}")

df.columns = df.columns.str.strip()
df = df.dropna(subset=['lat', 'lng', 'pm2d5']).copy()

# Drop GPS dropout rows (lat or lng == 0)
before = len(df)
df = df[(df['lat'] != 0.0) & (df['lng'] != 0.0)].copy()
print(f"  Dropped {before - len(df)} GPS dropout rows (lat/lng=0)")
print(f"  Valid rows remaining: {len(df)}")

# Parse Timestamp: DD-MM-YY HH:MM:SS
df['datetime_full'] = pd.to_datetime(df['Timestamp'], dayfirst=True)
df['datetime_full'] = df['datetime_full'].apply(
    lambda x: x.replace(year=x.year + 100) if x.year < 2000 else x
)

df['hour_of_day']   = df['datetime_full'].dt.hour
df['minute_of_day'] = df['datetime_full'].dt.hour * 60 + df['datetime_full'].dt.minute
df['time_of_day']   = df['datetime_full'].dt.time
df = df.sort_values('datetime_full').reset_index(drop=True)

print(f"  Date range : {df['datetime_full'].min()} -> {df['datetime_full'].max()}")

LAT_MIN, LAT_MAX = df['lat'].min(), df['lat'].max()
LON_MIN, LON_MAX = df['lng'].min(), df['lng'].max()

lat_pad      = (LAT_MAX - LAT_MIN) * 0.05
lon_pad      = (LON_MAX - LON_MIN) * 0.05
PLOT_LAT_MIN = LAT_MIN - lat_pad
PLOT_LAT_MAX = LAT_MAX + lat_pad
PLOT_LON_MIN = LON_MIN - lon_pad
PLOT_LON_MAX = LON_MAX + lon_pad

print(f"  Lat range  : {LAT_MIN:.6f} -> {LAT_MAX:.6f}")
print(f"  Lon range  : {LON_MIN:.6f} -> {LON_MAX:.6f}")

# =====================================================================
# STEP 2: CHUNKED SPATIO-TEMPORAL GROUPING
# =====================================================================
print("\n" + "="*70)
print("[STEP 2] Reference-point spatio-temporal grouping (chunked)")
print("="*70)

n       = len(df)
avg_lat = (LAT_MIN + LAT_MAX) / 2.0
scale   = np.array([111000.0, 111000.0 * np.cos(np.radians(avg_lat))])

coords_m = df[['lat', 'lng']].values * scale
tod      = df['minute_of_day'].values

n_chunks = int(np.ceil(n / CHUNK_SIZE))
print(f"  Total points : {n:,}")
print(f"  Chunk size   : {CHUNK_SIZE:,}")
print(f"  Total chunks : {n_chunks}")
print(f"  Peak RAM/chunk ~ {CHUNK_SIZE * n * 8 / 1e9:.2f} GB  "
      f"(full matrix = {n*n*8/1e9:.1f} GB)\n")

# Store each ref point's raw member set + its ref index for audit
raw_candidate_list = []   # list of (ref_idx, frozenset)

for chunk_idx in range(n_chunks):
    start = chunk_idx * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, n)

    chunk_coords = coords_m[start:end]
    diff         = chunk_coords[:, np.newaxis, :] - coords_m[np.newaxis, :, :]
    spatial_dist = np.sqrt((diff ** 2).sum(axis=2))
    del diff

    chunk_tod = tod[start:end]
    tdist_raw = np.abs(chunk_tod[:, np.newaxis] - tod[np.newaxis, :])
    time_dist = np.minimum(tdist_raw, 1440 - tdist_raw)
    del tdist_raw

    both_ok = (spatial_dist <= SPATIAL_BUFFER_M) & (time_dist <= TEMPORAL_WINDOW_MIN)
    del spatial_dist, time_dist

    for local_i in range(end - start):
        global_i = start + local_i
        members  = frozenset(np.where(both_ok[local_i])[0])
        if len(members) >= MIN_GROUP_SIZE:
            raw_candidate_list.append((global_i, members))

    del both_ok

    if (chunk_idx + 1) % 5 == 0 or chunk_idx == n_chunks - 1:
        print(f"  Chunk {chunk_idx+1:>4}/{n_chunks} done  |  "
              f"candidates so far: {len(raw_candidate_list):,}")

candidate_sets = [m for _, m in raw_candidate_list]
print(f"\n  Raw candidate groups : {len(candidate_sets):,}")

# ── Deduplication ─────────────────────────────────────────────────────
# For audit: track which ref points produced each unique set
frozenset_to_refs = {}
for ref_idx, members in raw_candidate_list:
    key = members
    if key not in frozenset_to_refs:
        frozenset_to_refs[key] = []
    frozenset_to_refs[key].append(ref_idx)

unique_groups = list(frozenset_to_refs.keys())
print(f"  After deduplication  : {len(unique_groups):,}")

# ── Bounded subset removal ────────────────────────────────────────────
print("  Removing subsets (3x size-bounded absorption)...")

final_groups   = []
absorbed_audit = []   # for verification Excel

for g in unique_groups:
    g_size   = len(g)
    absorber = None
    for other in unique_groups:
        if other is not g and g < other and len(other) <= 3 * g_size:
            absorber = other
            break
    if absorber is None:
        final_groups.append(g)
    else:
        absorbed_audit.append({
            'removed_set_size':  g_size,
            'removed_members_sample': str(sorted(g)[:10]),
            'absorber_set_size': len(absorber),
            'absorber_members_sample': str(sorted(absorber)[:10]),
            'size_ratio': round(len(absorber) / g_size, 2),
        })

print(f"  After bounded subset removal       : {len(final_groups):,}")

valid_groups = {idx: sorted(members) for idx, members in enumerate(final_groups)}
print(f"\n  Final valid group count: {len(valid_groups):,}")

# =====================================================================
# STEP 3: AGGREGATE GROUPS
# =====================================================================
print("\n[STEP 3] Aggregating group statistics...")

group_list = []
for gid, members in valid_groups.items():
    sub    = df.loc[members]
    size   = len(members)
    lat_c  = sub['lat'].mean()
    lon_c  = sub['lng'].mean()
    hour_b = int(sub['hour_of_day'].mode().iloc[0])
    t_ctr  = sub['minute_of_day'].mean()
    pm25_m = sub['pm2d5'].mean()
    pm10_m = sub['pm10'].mean()
    pres_m = sub['pressure'].mean() if 'pressure' in sub.columns else np.nan
    t_span = sub['minute_of_day'].max() - sub['minute_of_day'].min()

    sub_coords = sub[['lat', 'lng']].values * scale
    if size > 1:
        d         = sub_coords[:, np.newaxis, :] - sub_coords[np.newaxis, :, :]
        sp_spread = np.sqrt((d**2).sum(axis=2)).max()
    else:
        sp_spread = 0.0

    group_list.append({
        'group_id':         gid,
        'size':             size,
        'lat_center':       lat_c,
        'lon_center':       lon_c,
        'hour_bin':         hour_b,
        'time_center_min':  t_ctr,
        'hour_label':       f"{hour_b:02d}:00",
        'pm25_mean':        pm25_m,
        'pm10_mean':        pm10_m,
        'pressure_mean':    pres_m,
        'spatial_spread_m': sp_spread,
        'time_span_min':    t_span,
    })

groups_df = pd.DataFrame(group_list)
print(f"  Total groups : {len(groups_df):,}")
print(f"  Group size   min:{groups_df['size'].min()}  max:{groups_df['size'].max()}  "
      f"mean:{groups_df['size'].mean():.1f}")
print(f"  PM2.5 (group means)  min:{groups_df['pm25_mean'].min():.1f}  "
      f"max:{groups_df['pm25_mean'].max():.1f}  mean:{groups_df['pm25_mean'].mean():.1f} ug/m3")

# =====================================================================
# STEP 4: HOURLY BINNING SUMMARY
# =====================================================================
print("\n[STEP 4] Hourly distribution...")
unique_hours = sorted(groups_df['hour_bin'].unique())
for h in unique_hours:
    cnt = (groups_df['hour_bin'] == h).sum()
    print(f"  {h:02d}:00 -- {cnt:>5} groups")

# =====================================================================
# STEP 5: VISUALISATION
# =====================================================================
print("\n[STEP 5] Generating hourly plots...")

os.makedirs(OUTPUT_JPG_PATH, exist_ok=True)

vmin = groups_df['pm25_mean'].quantile(0.05)
vmax = groups_df['pm25_mean'].quantile(0.95)

saved_files = []
for hour in unique_hours:
    hour_data = groups_df[groups_df['hour_bin'] == hour]

    lat_range = PLOT_LAT_MAX - PLOT_LAT_MIN
    lon_range = PLOT_LON_MAX - PLOT_LON_MIN
    fig_w = 10
    fig_h = min(max(7, fig_w * (lat_range / lon_range)), 14)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sc = ax.scatter(
        hour_data['lon_center'], hour_data['lat_center'],
        c=hour_data['pm25_mean'],
        s=MARKER_SIZE,
        cmap='RdYlGn_r', alpha=0.75,
        edgecolors='black', linewidths=0.5,
        vmin=vmin, vmax=vmax, zorder=3
    )

    ax.set_xlim(PLOT_LON_MIN, PLOT_LON_MAX)
    ax.set_ylim(PLOT_LAT_MIN, PLOT_LAT_MAX)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(fontsize=8)

    ax.set_xlabel('Longitude', fontweight='bold', fontsize=11)
    ax.set_ylabel('Latitude',  fontweight='bold', fontsize=11)
    ax.set_title(
        f'PM2.5 Spatial Distribution -- Hour {hour:02d}:00\n'
        f'(all days aggregated | {len(hour_data)} groups | '
        f'buffer: {SPATIAL_BUFFER_M} m | window: +/-{TEMPORAL_WINDOW_MIN} min)',
        fontweight='bold', fontsize=12, pad=12
    )
    ax.grid(True, alpha=0.3, linestyle='--', zorder=1)

    cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('PM2.5 (ug/m3)', fontweight='bold', fontsize=10)

    stats = (f"N groups  : {len(hour_data)}\n"
             f"PM2.5 mean: {hour_data['pm25_mean'].mean():.1f} ug/m3\n"
             f"PM2.5 max : {hour_data['pm25_mean'].max():.1f} ug/m3\n"
             f"PM2.5 min : {hour_data['pm25_mean'].min():.1f} ug/m3")
    ax.text(0.02, 0.98, stats, transform=ax.transAxes, fontsize=8,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.55))

    plt.tight_layout()
    fname = os.path.join(OUTPUT_JPG_PATH, f"Air_Quality_Hour_{hour:02d}00.jpg")
    plt.savefig(fname, dpi=300, bbox_inches='tight', format='jpg')
    plt.close()
    saved_files.append(fname)
    print(f"  Saved: {fname}  ({len(hour_data)} groups)")

# =====================================================================
# STEP 6: MAIN EXCEL OUTPUT (all 6 sheets)
# =====================================================================
print("\n[STEP 6] Exporting main Excel...")

os.makedirs(OUTPUT_EXCEL_PATH, exist_ok=True)
excel_file = os.path.join(OUTPUT_EXCEL_PATH, "MobileMonitoring_SpatioTemporal.xlsx")

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:

    # Sheet 1: Group Summary
    gs = groups_df[[
        'group_id','size','hour_label',
        'lat_center','lon_center','time_center_min',
        'pm25_mean','pm10_mean','pressure_mean',
        'spatial_spread_m','time_span_min'
    ]].copy()
    gs.columns = [
        'Group_ID','Group_Size','Hour_Bin',
        'Latitude_Center','Longitude_Center','Time_Center_min',
        'PM25_Mean_ugm3','PM10_Mean_ugm3','Pressure_Mean_hPa',
        'Spatial_Spread_m','Time_Span_min'
    ]
    gs.sort_values('PM25_Mean_ugm3', ascending=False).to_excel(
        writer, sheet_name='Group_Summary', index=False)

    # Sheet 2: Hourly Statistics
    hourly_rows = []
    for h in unique_hours:
        hd = groups_df[groups_df['hour_bin'] == h]
        hourly_rows.append({
            'Hour':               f'{h:02d}:00',
            'Number_of_Groups':   len(hd),
            'Total_Observations': hd['size'].sum(),
            'PM25_Mean':          hd['pm25_mean'].mean(),
            'PM25_Min':           hd['pm25_mean'].min(),
            'PM25_Max':           hd['pm25_mean'].max(),
            'PM25_Std':           hd['pm25_mean'].std(),
            'PM10_Mean':          hd['pm10_mean'].mean(),
            'Pressure_Mean':      hd['pressure_mean'].mean(),
            'Avg_Group_Size':     hd['size'].mean(),
        })
    pd.DataFrame(hourly_rows).to_excel(
        writer, sheet_name='Hourly_Statistics', index=False)

    # Sheet 3: Top 20 Hotspots
    hot = groups_df.nlargest(20, 'pm25_mean')[[
        'group_id','size','hour_label',
        'lat_center','lon_center',
        'pm25_mean','pm10_mean','pressure_mean'
    ]].copy()
    hot.columns = [
        'Group_ID','Group_Size','Hour',
        'Latitude','Longitude',
        'PM25_ugm3','PM10_ugm3','Pressure_hPa'
    ]
    hot.to_excel(writer, sheet_name='Top_20_Hotspots', index=False)

    # Sheet 4: Original Data with Group Assignments
    df_out = df.copy()
    df_out['Group_ID'] = -1
    df_out['Hour_Bin'] = -1
    for gid, members in valid_groups.items():
        hb = groups_df.loc[groups_df['group_id'] == gid, 'hour_bin'].values[0]
        df_out.loc[members, 'Group_ID'] = gid
        df_out.loc[members, 'Hour_Bin'] = hb

    cols_out = ['datetime_full','lat','lng','pm2d5','pm10','pressure','Group_ID','Hour_Bin','time_of_day']
    cols_out = [c for c in cols_out if c in df_out.columns]
    df_out_final = df_out[cols_out].copy()
    df_out_final.columns = [
        'DateTime','Latitude','Longitude','PM25','PM10','Pressure',
        'Group_ID','Hour_Bin','Time_of_Day'
    ][:len(cols_out)]
    df_out_final.sort_values('DateTime').to_excel(
        writer, sheet_name='Original_Data_Grouped', index=False)

    # Sheet 5: Summary Statistics
    in_groups = groups_df['size'].sum()
    summary = pd.DataFrame({
        'Metric': [
            'Total measurements', 'Date range', 'Total valid groups',
            'Measurements in groups', 'Measurements NOT in any group',
            'Spatial buffer (m)', 'Temporal window (+/-min)', 'Min group size', '',
            'PM2.5 mean raw (ug/m3)', 'PM2.5 std raw', 'PM2.5 min raw', 'PM2.5 max raw',
            'PM2.5 25th pct raw', 'PM2.5 median raw', 'PM2.5 75th pct raw', '',
            'PM2.5 mean group-means', 'PM2.5 std group-means', '',
            'Unique hours analysed', 'Max group spatial spread (m)', 'Max group time span (min)',
        ],
        'Value': [
            len(df),
            f"{df['datetime_full'].min().date()} to {df['datetime_full'].max().date()}",
            len(groups_df), in_groups, len(df) - in_groups,
            SPATIAL_BUFFER_M, TEMPORAL_WINDOW_MIN, MIN_GROUP_SIZE, '',
            f"{df['pm2d5'].mean():.2f}", f"{df['pm2d5'].std():.2f}",
            f"{df['pm2d5'].min():.2f}", f"{df['pm2d5'].max():.2f}",
            f"{df['pm2d5'].quantile(0.25):.2f}", f"{df['pm2d5'].median():.2f}",
            f"{df['pm2d5'].quantile(0.75):.2f}", '',
            f"{groups_df['pm25_mean'].mean():.2f}", f"{groups_df['pm25_mean'].std():.2f}", '',
            len(unique_hours),
            f"{groups_df['spatial_spread_m'].max():.1f}",
            f"{groups_df['time_span_min'].max():.1f}",
        ]
    })
    summary.to_excel(writer, sheet_name='Summary_Statistics', index=False, header=False)

print(f"  Main Excel saved: {excel_file}")

# =====================================================================
# STEP 7: VERIFICATION EXCEL (dedup + subset audit trail)
# =====================================================================
print("\n[STEP 7] Exporting verification Excel...")

verify_file = os.path.join(OUTPUT_EXCEL_PATH, "Verification_Dedup_Subset.xlsx")

with pd.ExcelWriter(verify_file, engine='openpyxl') as writer:

    # ── Sheet A: Dedup sample ─────────────────────────────────────────
    # Show 50 frozensets that had multiple reference points → were collapsed to 1
    dedup_rows = []
    count = 0
    for fs, refs in frozenset_to_refs.items():
        if len(refs) > 1:       # only show groups that were actually duplicated
            members_sorted = sorted(fs)
            dedup_rows.append({
                'Unique_Group_Members_Sample': str(members_sorted[:15]),
                'Group_Size': len(fs),
                'N_Ref_Points_That_Produced_This_Set': len(refs),
                'Ref_Point_Indices_Sample': str(refs[:10]),
                'Action': 'Kept once (duplicates discarded)',
            })
            count += 1
            if count >= 50:
                break

    # Also add a few non-duplicated ones for comparison
    for fs, refs in frozenset_to_refs.items():
        if len(refs) == 1:
            members_sorted = sorted(fs)
            dedup_rows.append({
                'Unique_Group_Members_Sample': str(members_sorted[:15]),
                'Group_Size': len(fs),
                'N_Ref_Points_That_Produced_This_Set': len(refs),
                'Ref_Point_Indices_Sample': str(refs[:10]),
                'Action': 'Unique — kept',
            })
            count += 1
            if count >= 70:
                break

    pd.DataFrame(dedup_rows).to_excel(
        writer, sheet_name='Dedup_Sample_50', index=False)

    # ── Sheet B: Subset removal sample ───────────────────────────────
    # Show up to 100 groups that were absorbed + their absorber
    if absorbed_audit:
        pd.DataFrame(absorbed_audit[:100]).to_excel(
            writer, sheet_name='Subset_Removed_Sample_100', index=False)
    else:
        pd.DataFrame({'Note': ['No groups were absorbed in this run']}).to_excel(
            writer, sheet_name='Subset_Removed_Sample_100', index=False)

    # ── Sheet C: Survived groups sample ──────────────────────────────
    # Show 50 groups that survived both stages with their member details
    survived_rows = []
    for gid, members in list(valid_groups.items())[:50]:
        sub = df.loc[members]
        survived_rows.append({
            'Group_ID':           gid,
            'Group_Size':         len(members),
            'Member_Indices_Sample': str(members[:10]),
            'Hour_Bin':           f"{int(sub['hour_of_day'].mode().iloc[0]):02d}:00",
            'Lat_Center':         round(sub['lat'].mean(), 6),
            'Lon_Center':         round(sub['lng'].mean(), 6),
            'PM25_Mean':          round(sub['pm2d5'].mean(), 2),
            'PM10_Mean':          round(sub['pm10'].mean(), 2),
            'Time_Range_min':     f"{sub['minute_of_day'].min()} - {sub['minute_of_day'].max()}",
            'Date_Range':         f"{sub['datetime_full'].min().date()} to {sub['datetime_full'].max().date()}",
        })
    pd.DataFrame(survived_rows).to_excel(
        writer, sheet_name='Survived_Groups_Sample_50', index=False)

    # ── Sheet D: Counts summary ───────────────────────────────────────
    counts = pd.DataFrame({
        'Stage': [
            'Raw candidate groups (one per qualifying ref point)',
            'After deduplication (identical frozensets collapsed)',
            'Eliminated by deduplication',
            'After bounded subset removal (3x size cap)',
            'Eliminated by subset removal',
            'Final valid groups used for plotting',
        ],
        'Count': [
            len(raw_candidate_list),
            len(unique_groups),
            len(raw_candidate_list) - len(unique_groups),
            len(final_groups),
            len(unique_groups) - len(final_groups),
            len(valid_groups),
        ],
        'Notes': [
            'Every point within buffer produces a candidate set',
            'Python set() collapses identical member sets',
            'Pure duplicates — same members, different ref point',
            'Groups not fully contained within a 3x-larger group',
            'Subsets absorbed into slightly-larger neighbours only',
            'These are the dots you see on the hourly JPG maps',
        ]
    })
    counts.to_excel(writer, sheet_name='Stage_Counts', index=False)

print(f"  Verification Excel saved: {verify_file}")

# =====================================================================
# STEP 8: FINAL CONSOLE REPORT
# =====================================================================
print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print(f"""
  Total measurements     : {len(df):,}
  Date range             : {df['datetime_full'].min().date()} -> {df['datetime_full'].max().date()}
  Valid groups formed     : {len(valid_groups):,}
  Measurements in groups : {groups_df['size'].sum():,} / {len(df):,} ({100*groups_df['size'].sum()/len(df):.1f}%)
  Unique hours plotted    : {', '.join(f'{h:02d}:00' for h in unique_hours)}

  PM2.5 raw   mean:{df['pm2d5'].mean():.1f}  std:{df['pm2d5'].std():.1f}  [{df['pm2d5'].min():.1f}, {df['pm2d5'].max():.1f}] ug/m3
  PM2.5 grps  mean:{groups_df['pm25_mean'].mean():.1f}  std:{groups_df['pm25_mean'].std():.1f}  [{groups_df['pm25_mean'].min():.1f}, {groups_df['pm25_mean'].max():.1f}] ug/m3

  Top 5 most-polluted groups:""")

for rank, (_, row) in enumerate(groups_df.nlargest(5, 'pm25_mean').iterrows(), 1):
    print(f"    {rank}. PM2.5 {row['pm25_mean']:.1f} ug/m3 | "
          f"({row['lat_center']:.6f}, {row['lon_center']:.6f}) | "
          f"Hour {row['hour_label']} | n={int(row['size'])}")

print(f"""
  OUTPUT FILES
  -------------------------------------------------------
  Hourly JPGs ({len(saved_files)} files)   : {OUTPUT_JPG_PATH}
  Main Excel (6 sheets)     : {excel_file}
    Sheet 1 -- Group_Summary
    Sheet 2 -- Hourly_Statistics
    Sheet 3 -- Top_20_Hotspots
    Sheet 4 -- Original_Data_Grouped
    Sheet 5 -- Summary_Statistics
  Verification Excel (4 sheets) : {verify_file}
    Sheet A -- Dedup_Sample_50
    Sheet B -- Subset_Removed_Sample_100
    Sheet C -- Survived_Groups_Sample_50
    Sheet D -- Stage_Counts
""")
print("="*70 + "\n")


SPATIO-TEMPORAL CLUSTERING ANALYSIS — INITIALISATION

[STEP 1] Loading and preprocessing data...
  Loading: D:/LAMP/Vehicle_Monitoring/Data_File/Data_MobileMonitoring.xlsx
  Shape  : 70994 rows x 12 columns
  Columns: devId, Timestamp, temp, hum, pressure, pm2d5, pm10, aqi, lat, lng, grid, region
  Dropped 1250 GPS dropout rows (lat/lng=0)
  Valid rows remaining: 69744
  Date range : 2026-03-10 14:02:08 -> 2026-04-10 19:57:09
  Lat range  : 19.069450 -> 19.147141
  Lon range  : 72.860382 -> 72.936821

[STEP 2] Reference-point spatio-temporal grouping (chunked)
  Total points : 69,744
  Chunk size   : 3,000
  Total chunks : 24
  Peak RAM/chunk ~ 1.67 GB  (full matrix = 38.9 GB)

  Chunk    5/24 done  |  candidates so far: 14,898
  Chunk   10/24 done  |  candidates so far: 29,820
  Chunk   15/24 done  |  candidates so far: 43,933
  Chunk   20/24 done  |  candidates so far: 58,771
  Chunk   24/24 done  |  candidates so far: 68,329

  Raw candidate groups : 68,329
  After deduplication  :

#### 4. Bug Free Script for Expost Excel files

In [16]:
"""
===================================================================
SPATIO-TEMPORAL CLUSTERING ANALYSIS - Mobile Air Quality Sensor
===================================================================
BUGS FIXED IN THIS VERSION:
  BUG 1 (subset removal): frozenset < frozenset identity comparison
         was incorrectly flagging non-subsets as subsets.
         Fix: explicit issubset() check on frozenset objects.
  BUG 2 (Original_Data_Grouped export): later groups were
         overwriting Group_ID of rows already assigned by earlier
         groups. Fix: assign only if Group_ID is still -1.
===================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION
# =====================================================================
INPUT_FOLDER    = r"D:/LAMP/Vehicle_Monitoring/Data_File/"
OUTPUT_JPG_PATH = r"C:/Users/Atique/LAMP_Coding/Mobile_Monitoring/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"
OUTPUT_EXCEL_PATH = r"C:/Users/Atique/LAMP_Coding/Mobile_Monitoring/Spatio_Temporal_Analysis/Reference_Groyping_Mar18/"

SPATIAL_BUFFER_M    = 100
TEMPORAL_WINDOW_MIN = 30
MIN_GROUP_SIZE      = 2
CHUNK_SIZE          = 3000
MARKER_SIZE         = 40

# =====================================================================
print("\n" + "="*70)
print("SPATIO-TEMPORAL CLUSTERING ANALYSIS")
print("="*70)

# =====================================================================
# STEP 1: LOAD AND PREPROCESS
# =====================================================================
print("\n[STEP 1] Loading and preprocessing data...")

excel_files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(('.xlsx', '.xls', '.csv'))
]
if not excel_files:
    print("ERROR: No files found in input folder!")
    exit()

data_path = excel_files[0]
print(f"  Loading: {data_path}")
df = (pd.read_excel(data_path)
      if data_path.endswith(('.xlsx', '.xls'))
      else pd.read_csv(data_path))

print(f"  Shape  : {df.shape[0]} rows x {df.shape[1]} columns")
df.columns = df.columns.str.strip()
df = df.dropna(subset=['lat', 'lng', 'pm2d5']).copy()

before = len(df)
df = df[(df['lat'] != 0.0) & (df['lng'] != 0.0)].copy()
print(f"  Dropped {before - len(df)} GPS dropout rows")
print(f"  Valid rows: {len(df)}")

df['datetime_full'] = pd.to_datetime(df['Timestamp'], dayfirst=True)
df['datetime_full'] = df['datetime_full'].apply(
    lambda x: x.replace(year=x.year + 100) if x.year < 2000 else x
)
df['hour_of_day']   = df['datetime_full'].dt.hour
df['minute_of_day'] = df['datetime_full'].dt.hour * 60 + df['datetime_full'].dt.minute
df['time_of_day']   = df['datetime_full'].dt.time
df = df.sort_values('datetime_full').reset_index(drop=True)

print(f"  Date range: {df['datetime_full'].min()} -> {df['datetime_full'].max()}")

LAT_MIN, LAT_MAX = df['lat'].min(), df['lat'].max()
LON_MIN, LON_MAX = df['lng'].min(), df['lng'].max()
lat_pad = (LAT_MAX - LAT_MIN) * 0.05
lon_pad = (LON_MAX - LON_MIN) * 0.05
PLOT_LAT_MIN = LAT_MIN - lat_pad
PLOT_LAT_MAX = LAT_MAX + lat_pad
PLOT_LON_MIN = LON_MIN - lon_pad
PLOT_LON_MAX = LON_MAX + lon_pad
print(f"  Lat: {LAT_MIN:.6f} -> {LAT_MAX:.6f}")
print(f"  Lon: {LON_MIN:.6f} -> {LON_MAX:.6f}")

# =====================================================================
# STEP 2: CHUNKED SPATIO-TEMPORAL GROUPING
# =====================================================================
print("\n[STEP 2] Chunked spatio-temporal grouping...")

n       = len(df)
avg_lat = (LAT_MIN + LAT_MAX) / 2.0
scale   = np.array([111000.0, 111000.0 * np.cos(np.radians(avg_lat))])
coords_m = df[['lat', 'lng']].values * scale
tod      = df['minute_of_day'].values
n_chunks = int(np.ceil(n / CHUNK_SIZE))

print(f"  Points:{n:,}  Chunks:{n_chunks}  "
      f"Peak RAM/chunk~{CHUNK_SIZE*n*8/1e9:.2f}GB\n")

# Store (ref_idx, frozenset) for audit trail
raw_candidate_list = []

for chunk_idx in range(n_chunks):
    start = chunk_idx * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, n)

    chunk_coords = coords_m[start:end]
    diff         = chunk_coords[:, np.newaxis, :] - coords_m[np.newaxis, :, :]
    spatial_dist = np.sqrt((diff ** 2).sum(axis=2))
    del diff

    chunk_tod = tod[start:end]
    tdist_raw = np.abs(chunk_tod[:, np.newaxis] - tod[np.newaxis, :])
    time_dist = np.minimum(tdist_raw, 1440 - tdist_raw)
    del tdist_raw

    both_ok = (spatial_dist <= SPATIAL_BUFFER_M) & (time_dist <= TEMPORAL_WINDOW_MIN)
    del spatial_dist, time_dist

    for local_i in range(end - start):
        members = frozenset(np.where(both_ok[local_i])[0])
        if len(members) >= MIN_GROUP_SIZE:
            raw_candidate_list.append((start + local_i, members))

    del both_ok

    if (chunk_idx + 1) % 5 == 0 or chunk_idx == n_chunks - 1:
        print(f"  Chunk {chunk_idx+1:>4}/{n_chunks}  candidates so far: {len(raw_candidate_list):,}")

print(f"\n  Raw candidates : {len(raw_candidate_list):,}")

# ── Deduplication ──────────────────────────────────────────────────────
# Build a dict: frozenset -> list of ref_indices that produced it
frozenset_to_refs = {}
for ref_idx, members in raw_candidate_list:
    frozenset_to_refs.setdefault(members, []).append(ref_idx)

unique_groups = list(frozenset_to_refs.keys())   # all are frozenset objects
print(f"  After dedup    : {len(unique_groups):,}")

# ── Subset removal — FIXED ────────────────────────────────────────────
# BUG FIXED: use explicit .issubset() instead of < operator.
# The < operator on frozensets uses object identity checks internally
# in some comparison paths; .issubset() always does value comparison.
# Also fixed: compare every pair explicitly so no group is missed.
print("  Removing subsets (3x bounded, explicit issubset check)...")

# Sort by size ascending so smaller groups are checked first
unique_groups_sorted = sorted(unique_groups, key=len)
n_ug = len(unique_groups_sorted)

# Build a lookup: for each group, store its set() version for fast issubset
group_sets = [set(g) for g in unique_groups_sorted]
group_sizes = [len(g) for g in unique_groups_sorted]

absorbed_flags = [False] * n_ug
absorbed_audit = []

for i in range(n_ug):
    if absorbed_flags[i]:
        continue
    gi      = group_sets[i]
    gi_size = group_sizes[i]
    for j in range(i + 1, n_ug):         # j is always larger (sorted by size)
        gj_size = group_sizes[j]
        if gj_size > 3 * gi_size:
            break                         # sorted → no further j will qualify
        # CORRECT subset check: every element of gi must be in gj
        if gi_size < gj_size and gi.issubset(group_sets[j]):
            absorbed_flags[i] = True
            absorbed_audit.append({
                'Removed_Group_Size':       gi_size,
                'Removed_Members_Sample':   str(sorted(gi)[:15]),
                'Absorber_Group_Size':      gj_size,
                'Absorber_Members_Sample':  str(sorted(group_sets[j])[:15]),
                'Size_Ratio':               round(gj_size / gi_size, 2),
                'Is_True_Subset':           'YES — verified by issubset()',
            })
            break

final_groups = [frozenset(group_sets[i])
                for i in range(n_ug) if not absorbed_flags[i]]

print(f"  After subset removal: {len(final_groups):,}")

valid_groups = {idx: sorted(members) for idx, members in enumerate(final_groups)}
print(f"\n  Final valid groups: {len(valid_groups):,}")

# =====================================================================
# STEP 3: AGGREGATE
# =====================================================================
print("\n[STEP 3] Aggregating group statistics...")

group_list = []
for gid, members in valid_groups.items():
    sub    = df.loc[members]
    size   = len(members)
    lat_c  = sub['lat'].mean()
    lon_c  = sub['lng'].mean()
    hour_b = int(sub['hour_of_day'].mode().iloc[0])
    t_ctr  = sub['minute_of_day'].mean()
    pm25_m = sub['pm2d5'].mean()
    pm10_m = sub['pm10'].mean()
    pres_m = sub['pressure'].mean() if 'pressure' in sub.columns else np.nan
    t_span = sub['minute_of_day'].max() - sub['minute_of_day'].min()

    sub_coords = sub[['lat', 'lng']].values * scale
    if size > 1:
        d         = sub_coords[:, np.newaxis, :] - sub_coords[np.newaxis, :, :]
        sp_spread = np.sqrt((d ** 2).sum(axis=2)).max()
    else:
        sp_spread = 0.0

    group_list.append({
        'group_id':         gid,
        'size':             size,
        'lat_center':       lat_c,
        'lon_center':       lon_c,
        'hour_bin':         hour_b,
        'time_center_min':  t_ctr,
        'hour_label':       f"{hour_b:02d}:00",
        'pm25_mean':        pm25_m,
        'pm10_mean':        pm10_m,
        'pressure_mean':    pres_m,
        'spatial_spread_m': sp_spread,
        'time_span_min':    t_span,
    })

groups_df = pd.DataFrame(group_list)
print(f"  Groups:{len(groups_df):,}  "
      f"Size min/max/mean: {groups_df['size'].min()}/{groups_df['size'].max()}/{groups_df['size'].mean():.1f}  "
      f"PM2.5 mean:{groups_df['pm25_mean'].mean():.1f} ug/m3")

# =====================================================================
# STEP 4: HOURLY SUMMARY
# =====================================================================
print("\n[STEP 4] Hourly distribution...")
unique_hours = sorted(groups_df['hour_bin'].unique())
for h in unique_hours:
    print(f"  {h:02d}:00 -- {(groups_df['hour_bin']==h).sum():>5} groups")

# =====================================================================
# STEP 5: PLOTS
# =====================================================================
print("\n[STEP 5] Generating hourly plots...")
os.makedirs(OUTPUT_JPG_PATH, exist_ok=True)

vmin = groups_df['pm25_mean'].quantile(0.05)
vmax = groups_df['pm25_mean'].quantile(0.95)
saved_files = []

for hour in unique_hours:
    hour_data = groups_df[groups_df['hour_bin'] == hour]
    lat_range = PLOT_LAT_MAX - PLOT_LAT_MIN
    lon_range = PLOT_LON_MAX - PLOT_LON_MIN
    fig_w = 10
    fig_h = min(max(7, fig_w * (lat_range / lon_range)), 14)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sc = ax.scatter(
        hour_data['lon_center'], hour_data['lat_center'],
        c=hour_data['pm25_mean'], s=MARKER_SIZE,
        cmap='RdYlGn_r', alpha=0.75,
        edgecolors='black', linewidths=0.5,
        vmin=vmin, vmax=vmax, zorder=3
    )
    ax.set_xlim(PLOT_LON_MIN, PLOT_LON_MAX)
    ax.set_ylim(PLOT_LAT_MIN, PLOT_LAT_MAX)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.5f'))
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    ax.set_xlabel('Longitude', fontweight='bold', fontsize=11)
    ax.set_ylabel('Latitude',  fontweight='bold', fontsize=11)
    ax.set_title(
        f'PM2.5 Spatial Distribution -- Hour {hour:02d}:00\n'
        f'(all days aggregated | {len(hour_data)} groups | '
        f'buffer:{SPATIAL_BUFFER_M}m | window:+/-{TEMPORAL_WINDOW_MIN}min)',
        fontweight='bold', fontsize=12, pad=12
    )
    ax.grid(True, alpha=0.3, linestyle='--', zorder=1)
    cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('PM2.5 (ug/m3)', fontweight='bold', fontsize=10)
    stats = (f"N groups  : {len(hour_data)}\n"
             f"PM2.5 mean: {hour_data['pm25_mean'].mean():.1f}\n"
             f"PM2.5 max : {hour_data['pm25_mean'].max():.1f}\n"
             f"PM2.5 min : {hour_data['pm25_mean'].min():.1f}")
    ax.text(0.02, 0.98, stats, transform=ax.transAxes, fontsize=8,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.55))
    plt.tight_layout()
    fname = os.path.join(OUTPUT_JPG_PATH, f"Air_Quality_Hour_{hour:02d}00.jpg")
    plt.savefig(fname, dpi=300, bbox_inches='tight', format='jpg')
    plt.close()
    saved_files.append(fname)
    print(f"  Saved: Hour {hour:02d}:00  ({len(hour_data)} groups)")

# =====================================================================
# STEP 6: MAIN EXCEL (6 sheets)
# =====================================================================
print("\n[STEP 6] Exporting main Excel...")
os.makedirs(OUTPUT_EXCEL_PATH, exist_ok=True)
excel_file = os.path.join(OUTPUT_EXCEL_PATH, "MobileMonitoring_SpatioTemporal.xlsx")

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:

    # Sheet 1: Group Summary
    gs = groups_df[[
        'group_id','size','hour_label','lat_center','lon_center',
        'time_center_min','pm25_mean','pm10_mean','pressure_mean',
        'spatial_spread_m','time_span_min'
    ]].copy()
    gs.columns = [
        'Group_ID','Group_Size','Hour_Bin','Latitude_Center','Longitude_Center',
        'Time_Center_min','PM25_Mean_ugm3','PM10_Mean_ugm3','Pressure_Mean_hPa',
        'Spatial_Spread_m','Time_Span_min'
    ]
    gs.sort_values('PM25_Mean_ugm3', ascending=False).to_excel(
        writer, sheet_name='Group_Summary', index=False)

    # Sheet 2: Hourly Statistics
    hourly_rows = []
    for h in unique_hours:
        hd = groups_df[groups_df['hour_bin'] == h]
        hourly_rows.append({
            'Hour': f'{h:02d}:00',
            'N_Groups': len(hd),
            'Total_Obs': hd['size'].sum(),
            'PM25_Mean': round(hd['pm25_mean'].mean(), 2),
            'PM25_Min':  round(hd['pm25_mean'].min(), 2),
            'PM25_Max':  round(hd['pm25_mean'].max(), 2),
            'PM25_Std':  round(hd['pm25_mean'].std(), 2),
            'PM10_Mean': round(hd['pm10_mean'].mean(), 2),
            'Pressure_Mean': round(hd['pressure_mean'].mean(), 2),
            'Avg_Group_Size': round(hd['size'].mean(), 1),
        })
    pd.DataFrame(hourly_rows).to_excel(writer, sheet_name='Hourly_Statistics', index=False)

    # Sheet 3: Top 20 Hotspots
    hot = groups_df.nlargest(20, 'pm25_mean')[[
        'group_id','size','hour_label','lat_center','lon_center',
        'pm25_mean','pm10_mean','pressure_mean'
    ]].copy()
    hot.columns = ['Group_ID','Group_Size','Hour','Latitude','Longitude',
                   'PM25_ugm3','PM10_ugm3','Pressure_hPa']
    hot.to_excel(writer, sheet_name='Top_20_Hotspots', index=False)

    # Sheet 4: Original Data with Group Assignments — BUG FIXED
    # Build a clean mapping: row_index -> (group_id, hour_bin)
    # A row may technically belong to multiple groups (overlap is possible).
    # We assign it to the group where it first appears (lowest group_id).
    # This avoids the overwrite problem from the previous version.
    row_to_group = {}   # row_index -> group_id
    row_to_hour  = {}   # row_index -> hour_bin

    for gid, members in valid_groups.items():
        hb = groups_df.loc[groups_df['group_id'] == gid, 'hour_bin'].values[0]
        for m in members:
            if m not in row_to_group:          # assign only if not yet assigned
                row_to_group[m] = gid
                row_to_hour[m]  = hb

    df_out = df.copy()
    df_out['Group_ID'] = df_out.index.map(lambda i: row_to_group.get(i, -1))
    df_out['Hour_Bin'] = df_out.index.map(lambda i: row_to_hour.get(i,  -1))

    cols_out = ['datetime_full','lat','lng','pm2d5','pm10','Group_ID','Hour_Bin','time_of_day']
    if 'pressure' in df_out.columns:
        cols_out.insert(5, 'pressure')
    df_out[cols_out].sort_values('datetime_full').to_excel(
        writer, sheet_name='Original_Data_Grouped', index=True,
        index_label='Row_Index')

    # Sheet 5: Summary Statistics
    in_groups = groups_df['size'].sum()
    summary = pd.DataFrame({
        'Metric': [
            'Total measurements','Date range','Total valid groups',
            'Measurements assigned to a group',
            'Measurements not in any group (singletons)',
            'Spatial buffer (m)','Temporal window (+/-min)','Min group size','',
            'PM2.5 mean raw','PM2.5 std raw','PM2.5 min raw','PM2.5 max raw',
            'PM2.5 25th pct','PM2.5 median','PM2.5 75th pct','',
            'PM2.5 mean of group means','PM2.5 std of group means','',
            'Unique hours analysed',
            'Max group spatial spread (m)','Max group time span (min)',
        ],
        'Value': [
            len(df),
            f"{df['datetime_full'].min().date()} to {df['datetime_full'].max().date()}",
            len(groups_df), len(row_to_group), len(df) - len(row_to_group),
            SPATIAL_BUFFER_M, TEMPORAL_WINDOW_MIN, MIN_GROUP_SIZE, '',
            f"{df['pm2d5'].mean():.2f}", f"{df['pm2d5'].std():.2f}",
            f"{df['pm2d5'].min():.2f}", f"{df['pm2d5'].max():.2f}",
            f"{df['pm2d5'].quantile(0.25):.2f}", f"{df['pm2d5'].median():.2f}",
            f"{df['pm2d5'].quantile(0.75):.2f}", '',
            f"{groups_df['pm25_mean'].mean():.2f}",
            f"{groups_df['pm25_mean'].std():.2f}", '',
            len(unique_hours),
            f"{groups_df['spatial_spread_m'].max():.1f}",
            f"{groups_df['time_span_min'].max():.1f}",
        ]
    })
    summary.to_excel(writer, sheet_name='Summary_Statistics', index=False, header=False)

print(f"  Saved: {excel_file}")

# =====================================================================
# STEP 7: VERIFICATION EXCEL (audit trail)
# =====================================================================
print("\n[STEP 7] Exporting verification Excel...")
verify_file = os.path.join(OUTPUT_EXCEL_PATH, "Verification_Dedup_Subset.xlsx")

with pd.ExcelWriter(verify_file, engine='openpyxl') as writer:

    # Sheet A: Dedup sample — show groups with multiple ref points
    dedup_rows = []
    for fs, refs in list(frozenset_to_refs.items())[:200]:
        dedup_rows.append({
            'Members_Sample_(first_15)': str(sorted(fs)[:15]),
            'Full_Group_Size':           len(fs),
            'N_Ref_Points_Producing_This_Set': len(refs),
            'Ref_Point_Indices_Sample':  str(refs[:10]),
            'Is_Duplicate': 'YES' if len(refs) > 1 else 'NO (unique)',
            'Action': 'Kept once' if len(refs) > 1 else 'Kept (only one ref)',
            'Explanation': (
                f"{len(refs)} different reference points each independently "
                f"collected this exact same set of {len(fs)} members."
                if len(refs) > 1
                else "Only one reference point produced this set."
            )
        })
    pd.DataFrame(dedup_rows).to_excel(writer, sheet_name='Dedup_Sample', index=False)

    # Sheet B: Subset removed — now guaranteed to be true subsets
    if absorbed_audit:
        abs_df = pd.DataFrame(absorbed_audit[:200])
        abs_df['Verification_Note'] = (
            'Both groups shown. Removed group members are a TRUE subset '
            'of absorber members (confirmed by issubset()). '
            'Absorber is at most 3x the size of removed group.'
        )
        abs_df.to_excel(writer, sheet_name='Subset_Removed', index=False)
    else:
        pd.DataFrame({'Note': ['No groups absorbed']}).to_excel(
            writer, sheet_name='Subset_Removed', index=False)

    # Sheet C: Survived groups sample with their actual member rows
    survived_rows = []
    for gid, members in list(valid_groups.items())[:50]:
        sub = df.loc[members]
        survived_rows.append({
            'Group_ID':              gid,
            'Group_Size':            len(members),
            'All_Member_Row_Indices': str(members),
            'Hour_Bin':              f"{int(sub['hour_of_day'].mode().iloc[0]):02d}:00",
            'Lat_Center':            round(sub['lat'].mean(), 6),
            'Lon_Center':            round(sub['lng'].mean(), 6),
            'PM25_Mean':             round(sub['pm2d5'].mean(), 2),
            'PM10_Mean':             round(sub['pm10'].mean(), 2),
            'Time_Range_min':        f"{sub['minute_of_day'].min()}-{sub['minute_of_day'].max()}",
            'Date_Range':            f"{sub['datetime_full'].min().date()} to {sub['datetime_full'].max().date()}",
        })
    pd.DataFrame(survived_rows).to_excel(
        writer, sheet_name='Survived_Groups_Sample', index=False)

    # Sheet D: Stage counts
    pd.DataFrame({
        'Stage': [
            '1. Raw candidate groups',
            '2. After deduplication',
            '   Eliminated by dedup',
            '3. After subset removal',
            '   Eliminated by subset removal',
            '4. Final valid groups (plotted)',
        ],
        'Count': [
            len(raw_candidate_list),
            len(unique_groups),
            len(raw_candidate_list) - len(unique_groups),
            len(final_groups),
            len(unique_groups) - len(final_groups),
            len(valid_groups),
        ],
        'What_Was_Eliminated': [
            'Nothing yet',
            'Raw sets where identical frozensets were produced by multiple ref points',
            f"{len(raw_candidate_list)-len(unique_groups):,} redundant ref-point entries",
            'Groups that are true subsets of a group <=3x their size',
            f"{len(unique_groups)-len(final_groups):,} verified subsets removed",
            'These are the dots on the hourly JPG maps',
        ]
    }).to_excel(writer, sheet_name='Stage_Counts', index=False)

    # Sheet E: Group-size=2 lookup helper
    # For any size-2 group, show BOTH member row indices and their raw data
    size2_groups = {gid: members for gid, members in valid_groups.items()
                    if len(members) == 2}
    size2_rows = []
    for gid, members in list(size2_groups.items())[:500]:
        r0, r1 = members[0], members[1]
        row0   = df.loc[r0]
        row1   = df.loc[r1]
        size2_rows.append({
            'Group_ID':       gid,
            'Member_Row_0':   r0,
            'DateTime_0':     row0['datetime_full'],
            'Lat_0':          row0['lat'],
            'Lon_0':          row0['lng'],
            'PM25_0':         row0['pm2d5'],
            'Member_Row_1':   r1,
            'DateTime_1':     row1['datetime_full'],
            'Lat_1':          row1['lat'],
            'Lon_1':          row1['lng'],
            'PM25_1':         row1['pm2d5'],
            'Spatial_Dist_m': round(np.sqrt(((
                np.array([row0['lat'], row0['lng']]) -
                np.array([row1['lat'], row1['lng']])
            ) * scale)**2).sum(), 1),
            'Time_Diff_min':  abs(row0['minute_of_day'] - row1['minute_of_day']),
        })
    pd.DataFrame(size2_rows).to_excel(
        writer, sheet_name='Size2_Groups_Both_Members', index=False)

print(f"  Saved: {verify_file}")

# =====================================================================
# STEP 8: FINAL SUMMARY
# =====================================================================
print("\n" + "="*70)
print("COMPLETE")
print("="*70)
print(f"""
  Measurements      : {len(df):,}
  Valid groups       : {len(valid_groups):,}
  Assigned to groups : {len(row_to_group):,} / {len(df):,} rows
  Unique hours       : {', '.join(f'{h:02d}:00' for h in unique_hours)}
  PM2.5 raw mean     : {df['pm2d5'].mean():.1f} ug/m3

  Top 5 hotspot groups:""")
for rank, (_, row) in enumerate(groups_df.nlargest(5, 'pm25_mean').iterrows(), 1):
    print(f"    {rank}. {row['pm25_mean']:.1f} ug/m3 | "
          f"({row['lat_center']:.6f},{row['lon_center']:.6f}) | "
          f"Hour {row['hour_label']} | n={int(row['size'])}")
print(f"""
  Main Excel   : {excel_file}
  Verify Excel : {verify_file}
  JPG plots    : {OUTPUT_JPG_PATH}
""")
print("="*70)


SPATIO-TEMPORAL CLUSTERING ANALYSIS

[STEP 1] Loading and preprocessing data...
  Loading: D:/LAMP/Vehicle_Monitoring/Data_File/Data_MobileMonitoring.xlsx
  Shape  : 70994 rows x 12 columns
  Dropped 1250 GPS dropout rows
  Valid rows: 69744
  Date range: 2026-03-10 14:02:08 -> 2026-04-10 19:57:09
  Lat: 19.069450 -> 19.147141
  Lon: 72.860382 -> 72.936821

[STEP 2] Chunked spatio-temporal grouping...
  Points:69,744  Chunks:24  Peak RAM/chunk~1.67GB

  Chunk    5/24  candidates so far: 14,898
  Chunk   10/24  candidates so far: 29,820
  Chunk   15/24  candidates so far: 43,933
  Chunk   20/24  candidates so far: 58,771
  Chunk   24/24  candidates so far: 68,329

  Raw candidates : 68,329
  After dedup    : 42,104
  Removing subsets (3x bounded, explicit issubset check)...
  After subset removal: 23,767

  Final valid groups: 23,767

[STEP 3] Aggregating group statistics...
  Groups:23,767  Size min/max/mean: 2/576/51.0  PM2.5 mean:77.5 ug/m3

[STEP 4] Hourly distribution...
  00:00 -